# BraTS 2024 - Model Training

This notebook handles:
1. DataLoader initialization for preprocessed 2D slice tensors (`E:\abronikowski\data\processed`).
2. Model initialization for 2D **U-Net** (4 input channels: T1, T1CE, T2, FLAIR; 5 output classes: 0=background, 1=NCR, 2=ED, 3=ET, 4=RC).
3. Loss function setup (**DiceCELoss**) and optimizer (**AdamW**).
4. Training loop execution via the `Trainer` class with automatic checkpointing (`best_unet.pt`).
5. Visualization and export of learning curves (loss and validation Dice scores).

## 1. Environment Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Resolve repository root directory
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

repo_src = PROJECT_ROOT / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

import torch
import matplotlib.pyplot as plt

from brain_mri.data.dataset import create_dataloader
from brain_mri.models.unet import UNet
from brain_mri.models.losses import DiceCELoss
from brain_mri.training import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute device: {device}")
if device.type == "cuda":
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")

## 2. Initialize DataLoaders

> **Note for Windows users:** Recommended `num_workers` is 0 or 2. If you encounter any multiprocessing issues in the notebook, set `num_workers=0`.

In [ ]:
DATA_DIR = Path(r"E:\abronikowski\data")
train_dir = DATA_DIR / "processed" / "train"
val_dir = DATA_DIR / "processed" / "validate"

# Loader parameters
BATCH_SIZE = 16
NUM_WORKERS = 2  # Set to 0 if multiprocessing errors occur on Windows
PIN_MEMORY = torch.cuda.is_available()

train_loader = create_dataloader(
    split_dir=train_dir,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

val_loader = create_dataloader(
    split_dir=val_dir,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print(f"Train batches:      {len(train_loader)} (total samples: {len(train_loader.dataset)})")
print(f"Validation batches: {len(val_loader)} (total samples: {len(val_loader.dataset)})")

## 3. Model, Loss Function, and Trainer Initialization

In [ ]:
# U-Net: 4 input MRI modalities, 5 output classes (0: background, 1: NCR, 2: ED, 3: ET, 4: RC)
model = UNet(in_channels=4, num_classes=5, base_features=64)

criterion = DiceCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
checkpoint_dir = PROJECT_ROOT / "checkpoints"

trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=checkpoint_dir
)
print("Trainer initialized successfully.")

## 4. Training Loop Execution

In [ ]:
EPOCHS = 30
history = trainer.fit(epochs=EPOCHS)

## 5. Learning Curves Visualization

In [ ]:
plt.figure(figsize=(14, 5))

# Loss curves
plt.subplot(1, 2, 1)
plt.plot(history["train_loss"], label="Train Loss", linewidth=2)
plt.plot(history["val_loss"], label="Val Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss During Training")
plt.legend()
plt.grid(True)

# Validation Mean Dice curves
plt.subplot(1, 2, 2)
plt.plot(history["val_mean_dice"], label="Val Mean Dice", color="green", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Mean Dice Score")
plt.legend()
plt.grid(True)

plt.tight_layout()
curve_save_path = checkpoint_dir / "learning_curves.png"
plt.savefig(curve_save_path, dpi=150)
print(f"Saved curve plot to: {curve_save_path}")
plt.show()